# Oracyne AI — Voice Tutor Colab GPU Server

Run a fast streaming open-weight LLM (`qwen2.5:7b` via Ollama + Flash Attention + Q8 KV cache) and high-accuracy multilingual STT (`faster-whisper large-v3-turbo` + Silero VAD) on a **Google Colab L4 GPU (Colab Pro recommended)** or T4 GPU, exposed via a streaming Cloudflare Quick Tunnel.

### Quick Steps
1. In Colab, select **Runtime -> Change runtime type -> L4 GPU** (or **T4 GPU** — Antigravity automatically picks L4 on Colab Pro and falls back to T4 when needed).
2. Run Cells 1 to 4 in order.
3. Copy the printed **`https://....trycloudflare.com`** URL.
4. Open Oracyne AI **Settings** (`http://127.0.0.1:4142/settings`), choose **Google Colab (Remote GPU LLM)**, paste the URL, click **Test connection**, and save — then start a live voice call in **Coach** (`/coach`).

In [ ]:
#@title 1. Check GPU and install Ollama, Cloudflared, Gateway, and faster-whisper
!nvidia-smi || echo "No GPU detected — switch Runtime -> Change runtime type -> L4 GPU for fastest inference."
!apt-get -qq update && apt-get -qq install -y zstd pciutils lshw ffmpeg > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!pip -q install flask requests edge-tts faster-whisper
!pip -q install openai-whisper || echo "Optional openai-whisper backup skipped"
!ollama --version && cloudflared --version

In [ ]:
#@title 2. Choose model, Whisper STT toggle, and optional Bearer token
MODEL = "qwen2.5:7b"  #@param ["qwen2.5:7b", "llama3.1:8b", "gemma2:9b", "qwen2.5:14b"] {allow-input: true}
COLAB_API_KEY = ""  #@param {type:"string"}
ENABLE_WHISPER_STT = True  #@param {type:"boolean"}

print(f"Selected model: {MODEL}")
print("Whisper STT endpoint (/api/stt):", "ENABLED (faster-whisper large-v3-turbo)" if ENABLE_WHISPER_STT else "DISABLED")
print("Bearer token protection:", "ENABLED" if COLAB_API_KEY.strip() else "DISABLED (open quick-tunnel URL)")

In [ ]:
#@title 3. Start Ollama (Flash Attention + q8_0 KV + 60m keep-alive), pull model, and warm up GPU VRAM
import json
import os
import subprocess
import time
import urllib.request

subprocess.run(["pkill", "-f", "ollama serve"], check=False)
time.sleep(1)

env = os.environ.copy()
env["OLLAMA_HOST"] = "127.0.0.1:11434"
env["OLLAMA_ORIGINS"] = "*"
env["OLLAMA_KEEP_ALIVE"] = "60m"
env["OLLAMA_FLASH_ATTENTION"] = "1"
env["OLLAMA_KV_CACHE_TYPE"] = "q8_0"
env["OLLAMA_NUM_PARALLEL"] = "2"

log_file = open("/tmp/ollama.log", "w")
subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdin=subprocess.DEVNULL,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
    close_fds=True,
)

for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2) as resp:
            if resp.status == 200:
                print("Ollama server is running.")
                break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama failed to start. Check /tmp/ollama.log")

print(f"Ensuring {MODEL} is pulled (1-3 minutes the first time)...")
subprocess.run(["ollama", "pull", MODEL], check=True)

print("Warming up model in GPU VRAM...")
req = urllib.request.Request(
    "http://127.0.0.1:11434/api/chat",
    data=json.dumps({
        "model": MODEL,
        "stream": False,
        "format": "json",
        "keep_alive": "60m",
        "messages": [{"role": "user", "content": 'Reply with JSON: {"ready": true}'}],
    }).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(req, timeout=240) as resp:
    payload = json.loads(resp.read().decode("utf-8"))
    print("Warm-up response:", payload.get("message", {}).get("content", ""))

In [ ]:
#@title 4. Start Streaming Gateway v5 (/api/chat + faster-whisper large-v3-turbo /api/stt) & Cloudflare Tunnel
import json
import os
import re
import subprocess
import time
import urllib.request

GATEWAY_VERSION = "5"
GATEWAY_SCRIPT = '''import array
import asyncio
import base64
import ctypes
import glob
import hmac
import math
import os
import re
import site
import subprocess
import tempfile
import threading
import wave
import requests
from flask import Flask, Response, jsonify, request, stream_with_context

app = Flask(__name__)
OLLAMA_URL = "http://127.0.0.1:11434"
GATEWAY_VERSION = "5"
API_KEY = os.environ.get("ORACYNE_COLAB_API_KEY", "").strip()
ENABLE_WHISPER = os.environ.get("ORACYNE_ENABLE_WHISPER", "1") == "1"

_whisper_lock = threading.Lock()
_stt_infer_lock = threading.Lock()
_whisper_model = None
_whisper_checked = False
_whisper_engine = None
_whisper_model_name = None
_whisper_fp16 = False

_HALLUCINATION_PHRASES = {
    "thank you for watching",
    "thanks for watching",
    "please subscribe",
    "like and subscribe",
    "subscribe to my channel",
    "subtitles by",
    "translated by",
    "transcribed by",
    "captioning by",
    "amara.org",
    "ご視聴ありがとうございました",
    "字幕",
    "mbc 뉴스",
    "시청해주셔서 감사합니다",
    "terima kasih telah menonton",
    "terima kasih sudah menonton",
}


def _check_pkg_available(module_name: str) -> bool:
    try:
        __import__(module_name)
        return True
    except Exception:
        return False


_has_faster_whisper = ENABLE_WHISPER and _check_pkg_available("faster_whisper")
_has_openai_whisper = ENABLE_WHISPER and _check_pkg_available("whisper")
_whisper_available = _has_faster_whisper or _has_openai_whisper
_edge_tts_available = _check_pkg_available("edge_tts")


def _prepare_cuda_libs():
    lib_dirs = []
    for sp in site.getsitepackages():
        lib_dirs.extend(glob.glob(os.path.join(sp, "nvidia", "*", "lib")))
        torch_lib = os.path.join(sp, "torch", "lib")
        if os.path.isdir(torch_lib):
            lib_dirs.append(torch_lib)
    existing = [d for d in dict.fromkeys(lib_dirs) if os.path.isdir(d)]
    if existing:
        cur = os.environ.get("LD_LIBRARY_PATH", "")
        os.environ["LD_LIBRARY_PATH"] = ":".join(existing + ([cur] if cur else []))
        for d in existing:
            for pattern in ("libcudnn*.so*", "libcublas*.so*"):
                for so_path in sorted(glob.glob(os.path.join(d, pattern))):
                    try:
                        ctypes.CDLL(so_path, mode=ctypes.RTLD_GLOBAL)
                    except Exception:
                        pass


def _load_openai_whisper_locked():
    global _whisper_model, _whisper_engine, _whisper_model_name, _whisper_fp16
    import numpy as np
    import torch
    import whisper

    use_cuda = torch.cuda.is_available()
    device = "cuda" if use_cuda else "cpu"
    candidates = ("turbo", "small") if use_cuda else ("small",)
    tone = (0.15 * np.sin(2 * np.pi * 440 * np.arange(8000, dtype=np.float32) / 16000.0)).astype(np.float32)
    for name in candidates:
        try:
            model = whisper.load_model(name, device=device)
            model.transcribe(tone, fp16=bool(use_cuda), temperature=0.0, beam_size=1)
            _whisper_model = model
            _whisper_engine = "openai-whisper"
            _whisper_model_name = name
            _whisper_fp16 = bool(use_cuda)
            print(f"[stt] Loaded & warmed openai-whisper '{name}' on {device}", flush=True)
            return model
        except Exception as err:
            print(f"[stt] openai-whisper '{name}' failed ({err})", flush=True)
    _whisper_model = None
    _whisper_engine = None
    _whisper_model_name = None
    _whisper_fp16 = False
    return None


def _load_faster_whisper_locked():
    global _whisper_model, _whisper_engine, _whisper_model_name, _whisper_fp16
    _prepare_cuda_libs()
    import numpy as np
    import torch
    from faster_whisper import WhisperModel

    use_cuda = torch.cuda.is_available()
    device = "cuda" if use_cuda else "cpu"
    compute_candidates = ("float16", "int8_float16") if use_cuda else ("int8",)
    model_candidates = ("large-v3-turbo", "turbo", "small") if use_cuda else ("small",)
    tone = (0.15 * np.sin(2 * np.pi * 440 * np.arange(8000, dtype=np.float32) / 16000.0)).astype(np.float32)

    for name in model_candidates:
        for compute_type in compute_candidates:
            try:
                model = WhisperModel(name, device=device, compute_type=compute_type)
                # Warm CTranslate2 CUDA/cuDNN kernels on a non-zero tone first so any cuDNN mismatch surfaces at boot.
                segs, _ = model.transcribe(tone, beam_size=1, temperature=0.0, vad_filter=False)
                list(segs)
                try:
                    segs_vad, _ = model.transcribe(tone, beam_size=1, temperature=0.0, vad_filter=True)
                    list(segs_vad)
                except Exception as vad_err:
                    print(f"[stt] VAD warmup notice for {name}: {vad_err}", flush=True)
                _whisper_model = model
                _whisper_engine = "faster-whisper"
                _whisper_model_name = name
                _whisper_fp16 = bool(use_cuda)
                print(f"[stt] Loaded faster-whisper '{name}' on {device} ({compute_type})", flush=True)
                return model
            except Exception as err:
                print(f"[stt] faster-whisper '{name}' ({compute_type}) failed: {err}", flush=True)
    return None


def get_whisper_model():
    global _whisper_checked
    if not _whisper_available:
        return None
    with _whisper_lock:
        if _whisper_checked:
            return _whisper_model
        _whisper_checked = True
        if _has_faster_whisper:
            model = _load_faster_whisper_locked()
            if model is not None:
                return model
        if _has_openai_whisper:
            return _load_openai_whisper_locked()
        return None


def _is_16k_mono_pcm_wav(wav_path: str) -> bool:
    try:
        with wave.open(wav_path, "rb") as wf:
            return wf.getnchannels() == 1 and wf.getsampwidth() == 2 and wf.getframerate() == 16000
    except Exception:
        return False


def audio_has_speech(wav_path: str) -> bool:
    """Reject digital silence or near-zero background clips before invoking Whisper."""
    try:
        with wave.open(wav_path, "rb") as wf:
            if wf.getsampwidth() != 2:
                return True
            frames = wf.readframes(wf.getnframes())
        if not frames:
            return False
        samples = array.array("h")
        samples.frombytes(frames)
        if not samples:
            return False
        peak = max(abs(s) for s in samples) / 32768.0
        if peak < 0.008:
            return False
        rms = math.sqrt(sum(s * s for s in samples) / len(samples)) / 32768.0
        return rms >= 0.002
    except Exception:
        return True


def clean_stt_transcript(text: str) -> str:
    cleaned = re.sub(r"\\s+", " ", (text or "").strip())
    if not cleaned:
        return ""
    # Drop pure punctuation artifacts.
    if re.fullmatch(r"[\\W_]+", cleaned, flags=re.UNICODE):
        return ""
    norm = re.sub(r"[.!?,。！？、\\s]+$", "", cleaned.lower()).strip()
    if not norm or norm in _HALLUCINATION_PHRASES:
        return ""
    for phrase in ("thank you for watching", "thanks for watching", "subtitles by", "amara.org", "ご視聴ありがとうございました"):
        if norm == phrase or norm.startswith(phrase + " "):
            return ""
    return cleaned


def _transcribe_with_faster_whisper(model, audio_path: str, lang_param, prompt_param):
    transcribe_kwargs = {
        "language": lang_param,
        "initial_prompt": prompt_param,
        "beam_size": 5,
        "best_of": 5,
        "temperature": 0.0,
        "condition_on_previous_text": False,
        "no_speech_threshold": 0.6,
        "log_prob_threshold": -1.0,
        "compression_ratio_threshold": 2.4,
    }
    try:
        segments, info = model.transcribe(
            audio_path,
            vad_filter=True,
            vad_parameters={"min_silence_duration_ms": 550, "speech_pad_ms": 300, "threshold": 0.45},
            **transcribe_kwargs,
        )
        segment_list = list(segments)
    except Exception as vad_err:
        if "vad" in str(vad_err).lower() or "onnx" in str(vad_err).lower():
            segments, info = model.transcribe(audio_path, vad_filter=False, **transcribe_kwargs)
            segment_list = list(segments)
        else:
            raise

    kept = []
    for seg in segment_list:
        no_speech = float(getattr(seg, "no_speech_prob", 0.0) or 0.0)
        avg_logprob = float(getattr(seg, "avg_logprob", 0.0) or 0.0)
        if no_speech > 0.65 or avg_logprob < -1.15:
            continue
        piece = (getattr(seg, "text", "") or "").strip()
        if piece:
            kept.append(piece)

    transcript = clean_stt_transcript(" ".join(kept))
    detected_lang = getattr(info, "language", None) if info is not None else None
    return transcript, detected_lang


def _transcribe_with_openai_whisper(model, audio_path: str, lang_param, prompt_param):
    kwargs = {
        "fp16": _whisper_fp16,
        "temperature": 0.0,
        "beam_size": 5,
        "condition_on_previous_text": False,
        "no_speech_threshold": 0.6,
        "logprob_threshold": -1.0,
        "compression_ratio_threshold": 2.4,
    }
    if lang_param:
        kwargs["language"] = lang_param
    if prompt_param:
        kwargs["initial_prompt"] = prompt_param

    result = model.transcribe(audio_path, **kwargs)
    kept = []
    for seg in result.get("segments") or []:
        no_speech = float(seg.get("no_speech_prob", 0.0) or 0.0)
        avg_logprob = float(seg.get("avg_logprob", 0.0) or 0.0)
        if no_speech > 0.65 or avg_logprob < -1.15:
            continue
        piece = (seg.get("text") or "").strip()
        if piece:
            kept.append(piece)
    raw_text = " ".join(kept) if (result.get("segments") or kept) else (result.get("text") or "")
    transcript = clean_stt_transcript(raw_text)
    return transcript, result.get("language")


def check_auth():
    if not API_KEY:
        return True
    auth = request.headers.get("Authorization", "")
    return hmac.compare_digest(auth, f"Bearer {API_KEY}")


@app.route("/api/health", methods=["GET"])
def health():
    if not check_auth():
        return jsonify({"error": "Unauthorized"}), 401
    return jsonify({
        "status": "ok",
        "gatewayVersion": GATEWAY_VERSION,
        "whisperAvailable": _whisper_available,
        "whisperLoaded": _whisper_model is not None,
        "whisperEngine": _whisper_engine,
        "whisperModel": _whisper_model_name,
        "edgeTtsAvailable": _edge_tts_available,
    })


@app.route("/api/stt/warmup", methods=["POST", "GET"])
def stt_warmup():
    if not check_auth():
        return jsonify({"error": "Unauthorized"}), 401
    model = get_whisper_model()
    return jsonify({
        "loaded": model is not None,
        "engine": _whisper_engine,
        "model": _whisper_model_name,
        "fp16": _whisper_fp16,
    })


@app.route("/api/stt", methods=["POST"])
def stt():
    if not check_auth():
        return jsonify({"error": "Unauthorized"}), 401
    model = get_whisper_model()
    if model is None:
        return jsonify({"error": "Whisper STT is unavailable on this runtime."}), 501
    payload = request.get_json(silent=True) or {}
    audio_b64 = payload.get("audioBase64", "")
    if not audio_b64:
        return jsonify({"error": "Missing audioBase64"}), 400

    raw_lang = (payload.get("language") or "auto").strip().lower()
    lang_hint = raw_lang.split("-")[0]
    lang_param = lang_hint if (raw_lang not in ("", "auto", "mixed") and len(lang_hint) in (2, 3)) else None

    raw_prompt = (payload.get("prompt") or "").strip()
    # Only pass initial_prompt when language is locked so auto-detection stays unbiased.
    prompt_param = raw_prompt if (lang_param and raw_prompt) else None

    mime = (payload.get("mimeType") or "").lower()
    suffix = ".webm"
    if "wav" in mime:
        suffix = ".wav"
    elif "mp4" in mime or "m4a" in mime:
        suffix = ".m4a"
    elif "ogg" in mime:
        suffix = ".ogg"
    elif "mp3" in mime or "mpeg" in mime:
        suffix = ".mp3"

    raw_path = None
    wav_path = None
    try:
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
            tmp.write(base64.b64decode(audio_b64))
            tmp.flush()
            raw_path = tmp.name

        if suffix == ".wav" and _is_16k_mono_pcm_wav(raw_path):
            if not audio_has_speech(raw_path):
                return jsonify({
                    "transcript": "",
                    "detectedLanguage": None,
                    "provider": "colab-whisper",
                    "engine": _whisper_engine,
                    "model": _whisper_model_name,
                })
            audio_path = raw_path
        else:
            wav_path = f"{raw_path}.wav"
            ff = subprocess.run(
                ["ffmpeg", "-y", "-nostdin", "-loglevel", "error", "-i", raw_path, "-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le", wav_path],
                capture_output=True,
                timeout=15,
                check=False,
            )
            converted = ff.returncode == 0 and os.path.exists(wav_path) and os.path.getsize(wav_path) > 44
            if converted and not audio_has_speech(wav_path):
                return jsonify({
                    "transcript": "",
                    "detectedLanguage": None,
                    "provider": "colab-whisper",
                    "engine": _whisper_engine,
                    "model": _whisper_model_name,
                })

            audio_path = wav_path if converted else raw_path

        with _stt_infer_lock:
            if _whisper_engine == "faster-whisper":
                try:
                    transcript, detected_lang = _transcribe_with_faster_whisper(model, audio_path, lang_param, prompt_param)
                except Exception as fw_err:
                    print(f"[stt] faster-whisper runtime error ({fw_err}); falling back to openai-whisper", flush=True)
                    if _has_openai_whisper:
                        with _whisper_lock:
                            fallback_model = _load_openai_whisper_locked()
                        if fallback_model is not None:
                            transcript, detected_lang = _transcribe_with_openai_whisper(fallback_model, audio_path, lang_param, prompt_param)
                        else:
                            raise
                    else:
                        raise
            else:
                transcript, detected_lang = _transcribe_with_openai_whisper(model, audio_path, lang_param, prompt_param)

        return jsonify({
            "transcript": transcript,
            "detectedLanguage": detected_lang if transcript else None,
            "provider": "colab-whisper",
            "engine": _whisper_engine,
            "model": _whisper_model_name,
        })
    finally:
        for p in (raw_path, wav_path):
            if p and os.path.exists(p):
                try:
                    os.unlink(p)
                except Exception:
                    pass


@app.route("/api/tts", methods=["POST"])
def tts():
    if not check_auth():
        return jsonify({"error": "Unauthorized"}), 401
    if not _edge_tts_available:
        return jsonify({"error": "edge-tts is unavailable on this runtime."}), 501
    payload = request.get_json(silent=True) or {}
    text = (payload.get("text") or "").strip()
    if not text:
        return jsonify({"error": "Missing text"}), 400
    voice = (payload.get("voice") or "en-US-AvaMultilingualNeural").strip()
    rate = (payload.get("rate") or "+0%").strip()
    pitch = (payload.get("pitch") or "+0Hz").strip()

    async def synth_bytes():
        import edge_tts
        communicate = edge_tts.Communicate(text, voice=voice, rate=rate, pitch=pitch)
        chunks = []
        async for item in communicate.stream():
            if item.get("type") == "audio":
                chunks.append(item["data"])
        return b"".join(chunks)

    audio_bytes = asyncio.run(synth_bytes())
    if not audio_bytes:
        return jsonify({"error": "Empty TTS output"}), 502
    return jsonify({
        "audioBase64": base64.b64encode(audio_bytes).decode("ascii"),
        "provider": "colab-edge-tts",
        "voice": voice,
    })


@app.route("/<path:subpath>", methods=["GET", "POST", "OPTIONS"])
def proxy(subpath):
    if request.method == "OPTIONS":
        return ("", 204)
    if not check_auth():
        return jsonify({"error": "Unauthorized"}), 401
    target_url = f"{OLLAMA_URL}/{subpath}"
    headers = {"Content-Type": request.headers.get("Content-Type", "application/json")}
    upstream = requests.request(
        method=request.method,
        url=target_url,
        params=request.args,
        data=request.get_data(),
        headers=headers,
        stream=True,
        timeout=180,
    )

    def generate():
        try:
            for chunk in upstream.iter_content(chunk_size=64):
                if chunk:
                    yield chunk
        finally:
            upstream.close()

    resp_headers = {
        "Content-Type": upstream.headers.get("Content-Type", "application/json"),
        "Cache-Control": "no-cache",
        "X-Accel-Buffering": "no",
    }
    return Response(stream_with_context(generate()), status=upstream.status_code, headers=resp_headers)


if __name__ == "__main__":
    app.run(host="127.0.0.1", port=11435, threaded=True)
'''

with open("/tmp/oracyne_gateway.py", "w") as f:
    f.write(GATEWAY_SCRIPT)

secret = COLAB_API_KEY.strip()
verify_headers = {"Authorization": f"Bearer {secret}"} if secret else {}

reused_url = None
try:
    with open("/tmp/cloudflared.log", "r") as f:
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
    if m:
        candidate_url = m.group(0).rstrip("/")
        with urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:11435/api/health", headers=verify_headers), timeout=2) as h_resp:
            if h_resp.status == 200:
                h_data = json.loads(h_resp.read().decode("utf-8"))
                whisper_ok = (not h_data.get("whisperAvailable")) or bool(h_data.get("whisperLoaded"))
                if str(h_data.get("gatewayVersion")) == GATEWAY_VERSION and whisper_ok:
                    with urllib.request.urlopen(urllib.request.Request(f"{candidate_url}/api/tags", headers=verify_headers), timeout=4) as p_resp:
                        if p_resp.status == 200:
                            reused_url = candidate_url
except Exception:
    reused_url = None

if reused_url:
    print(f"Reusing active streaming gateway & tunnel: {reused_url}")
else:
    subprocess.run(["pkill", "-f", "oracyne_gateway.py"], check=False)
    subprocess.run(["pkill", "-f", "cloudflared"], check=False)
    time.sleep(1)

    gw_env = os.environ.copy()
    gw_env["ORACYNE_COLAB_API_KEY"] = secret
    gw_env["ORACYNE_ENABLE_WHISPER"] = "1" if ENABLE_WHISPER_STT else "0"

    gw_log = open("/tmp/oracyne_gateway.log", "w")
    subprocess.Popen(
        ["python3", "/tmp/oracyne_gateway.py"],
        env=gw_env,
        stdin=subprocess.DEVNULL,
        stdout=gw_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
        close_fds=True,
    )

    whisper_enabled = False
    for _ in range(25):
        try:
            req = urllib.request.Request("http://127.0.0.1:11435/api/health", headers=verify_headers)
            with urllib.request.urlopen(req, timeout=2) as resp:
                if resp.status == 200:
                    h = json.loads(resp.read().decode("utf-8"))
                    whisper_enabled = bool(h.get("whisperAvailable"))
                    print(f"Streaming gateway v{GATEWAY_VERSION} is running on port 11435.")
                    break
        except Exception:
            time.sleep(0.5)
    else:
        raise RuntimeError("Gateway failed to start. Check /tmp/oracyne_gateway.log")

    if whisper_enabled:
        print("Warming up faster-whisper large-v3-turbo in GPU VRAM...")
        w_req = urllib.request.Request(
            "http://127.0.0.1:11435/api/stt/warmup",
            data=b"{}",
            headers={"Content-Type": "application/json", **verify_headers},
            method="POST",
        )
        with urllib.request.urlopen(w_req, timeout=180) as w_resp:
            w_info = json.loads(w_resp.read().decode("utf-8"))
            print(f"Whisper STT ready: engine={w_info.get('engine')} model={w_info.get('model')} loaded={w_info.get('loaded')}")

    tunnel_log = open("/tmp/cloudflared.log", "w")
    subprocess.Popen(
        [
            "cloudflared",
            "tunnel",
            "--url",
            "http://127.0.0.1:11435",
            "--http-host-header",
            "localhost:11435",
            "--no-autoupdate",
        ],
        stdin=subprocess.DEVNULL,
        stdout=tunnel_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
        close_fds=True,
    )

public_url = reused_url
if not public_url:
    for _ in range(30):
        time.sleep(1)
        with open("/tmp/cloudflared.log", "r") as f:
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if match:
                public_url = match.group(0)
                break

if not public_url:
    with open("/tmp/cloudflared.log", "r") as f:
        print(f.read())
    raise RuntimeError("Could not extract trycloudflare.com URL.")

for _ in range(15):
    try:
        req = urllib.request.Request(f"{public_url}/api/tags", headers=verify_headers)
        with urllib.request.urlopen(req, timeout=5) as resp:
            tags = json.loads(resp.read().decode("utf-8"))
            pulled = [m.get("name") for m in tags.get("models", [])]
            print("\n" + "=" * 60)
            print("ORACYNE AI — GOOGLE COLAB SETTINGS")
            print("=" * 60)
            print("AI provider          : Google Colab (Remote GPU LLM)")
            print(f"Colab tunnel URL     : {public_url}")
            print(f"Model name           : {MODEL}")
            print("API format           : Ollama (/api/chat streaming)")
            print(f"Optional Bearer token: {'(set)' if secret else '(leave blank)'}")
            print(f"Pulled models        : {', '.join(pulled)}")
            print("=" * 60)
            print("Keep this Colab runtime active while practicing.")
            break
    except Exception:
        time.sleep(1)
else:
    print(f"Tunnel created at {public_url}, waiting for DNS propagation.")

### Optional Fallback: `ngrok` Tunnel
Only run the cell below if `trycloudflare.com` is blocked on your network. Requires a free token from https://dashboard.ngrok.com.

In [ ]:
#@title Optional: Expose via ngrok instead of Cloudflare
NGROK_AUTHTOKEN = ""  #@param {type:"string"}
if not NGROK_AUTHTOKEN.strip():
    raise ValueError("Set NGROK_AUTHTOKEN first, or use Cell 4 (Cloudflare) above.")

!pip -q install pyngrok
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
tunnel = ngrok.connect(11435, "http")
print("ngrok URL for Oracyne Settings:", tunnel.public_url)